# NB86 — The E₄–Metric Bridge

**Goal**: Bridge the two "flat" objects of the solenoid framework:
1. **The Cayley Laplacian** L (48×48) — algebraic sector, Ricci-flat (NB46), Tr(L) = 240 = c₁(E₄) (NB47)
2. **The solenoid metric** g̃_R⁻¹ (4×4) — geometric sector, Riemannian-flat (NB85), Tr = 94/15

Both encode {2,3,5,7} arithmetic. Both are flat. The question: how do they connect?

**Frontier target**: Step 6 of the gravity hierarchy derivation chain (5/6 proved):
M_Pl/M_Z = 240⁴ × 7⁹ = c₁(E₄)^{ω(P₄)} × p₄^{σ₃(p₁)}

**Sections**:
- §1: Reconstruct both objects; verify known invariants
- §2: Spectral moment ratios ρ_n = Tr(L^n)/c_n(E₄)
- §3: Primorial trace identities — 240 = P₃+P₄ and the metric connection
- §4: Heat kernel and spectral zeta at metric-related scales
- §5: The gravity hierarchy — testing the selection mechanism

In [2]:
import sys, io, contextlib
import numpy as np
from fractions import Fraction
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, RHO
print(f"P4 = {SA.P}, phi(P4) = {SA.PHI}, primes = {SA.primes}")
print(f"rho = 1/sqrt({SA.P}) = {RHO:.6f}")
print(f"|Z*_210| = {len(SA.Z_star)}")

# -- S1a: Build the Cayley Laplacian (48x48) --
# Separable generators (from NB45/46): one max-order gen per prime factor
# Z2 (p=3): gen 71; Z4 (p=5): gens 127,43; Z6 (p=7): gens 31,61
gen_set = [31, 43, 61, 71, 127]
print(f"\nSeparable generators: {gen_set} (|S|={len(gen_set)})")

Z = SA.Z_star
idx = {g: i for i, g in enumerate(Z)}
n = len(Z)
A = np.zeros((n, n))
for g in Z:
    for s in gen_set:
        gs = (g * s) % SA.P
        if gs in idx:
            A[idx[g], idx[gs]] = 1
L = np.diag(A.sum(axis=1)) - A

eigvals = np.sort(np.linalg.eigvalsh(L))
eigvals_int = np.round(eigvals).astype(int)

mult = Counter(eigvals_int)
print(f"\nCayley Laplacian eigenvalue spectrum:")
for k in sorted(mult):
    print(f"  lam = {k:2d}  mult = {mult[k]:2d}")
TrL = eigvals.sum()
TrL2 = (eigvals**2).sum()
print(f"\nTr(L)  = {TrL:.1f}")
print(f"Tr(L2) = {TrL2:.1f}")
print(f"Lam_max = {eigvals_int.max()}")

# -- S1b: Build the solenoid metric g_R_inv (4x4) --
primes = SA.primes  # [2, 3, 5, 7]
primorials = [1]
for p in primes:
    primorials.append(primorials[-1] * p)  # [1, 2, 6, 30, 210]

g_inv_exact = [[Fraction(0)] * 4 for _ in range(4)]
diag_f = [Fraction(1 + primes[k], primorials[k]) for k in range(4)]
sub_f  = [Fraction(-1, primorials[k]) for k in range(3)]

for k in range(4):
    g_inv_exact[k][k] = diag_f[k]
for k in range(3):
    g_inv_exact[k][k+1] = sub_f[k]
    g_inv_exact[k+1][k] = sub_f[k]

print("\n\n-- Solenoid metric g_R_inv (exact) --")
print("Diagonal:     ", [str(d) for d in diag_f])
print("Sub-diagonal: ", [str(s) for s in sub_f])
tr_ginv = sum(diag_f)
print(f"Tr(g_R_inv)  = {tr_ginv} = {float(tr_ginv):.6f}")

g_inv_float = np.array([[float(g_inv_exact[i][j]) for j in range(4)] for i in range(4)])
eigs_metric = np.sort(np.linalg.eigvalsh(g_inv_float))
print(f"Eigenvalues:  [{', '.join(f'{e:.4f}' for e in eigs_metric)}]")

# Determinant via tridiagonal recurrence
det_seq = [Fraction(0)] * 5
det_seq[0] = Fraction(1)
det_seq[1] = diag_f[0]
for k in range(1, 4):
    det_seq[k+1] = diag_f[k] * det_seq[k] - sub_f[k-1]**2 * det_seq[k-1]
det_ginv = det_seq[4]
print(f"det(g_R_inv) = {det_ginv} = {float(det_ginv):.6f}")

print("\n-- Summary --")
print(f"Cayley Laplacian: Tr(L) = {int(TrL)}, Ricci-flat (NB46)")
print(f"Solenoid metric:  Tr(g_R_inv) = {tr_ginv}, Riemannian-flat (NB85)")
print(f"Both objects encode {{2,3,5,7}} and are flat.")

P4 = 210, phi(P4) = 48, primes = [2, 3, 5, 7]
rho = 1/sqrt(210) = 0.069007
|Z*_210| = 48

Separable generators: [31, 43, 61, 71, 127] (|S|=5)

Cayley Laplacian eigenvalue spectrum:
  lam =  0  mult =  1
  lam =  1  mult =  2
  lam =  2  mult =  3
  lam =  3  mult =  8
  lam =  4  mult =  4
  lam =  5  mult = 12
  lam =  6  mult =  4
  lam =  7  mult =  8
  lam =  8  mult =  3
  lam =  9  mult =  2
  lam = 10  mult =  1

Tr(L)  = 240.0
Tr(L2) = 1440.0
Lam_max = 10


-- Solenoid metric g_R_inv (exact) --
Diagonal:      ['3', '2', '1', '4/15']
Sub-diagonal:  ['-1', '-1/2', '-1/6']
Tr(g_R_inv)  = 94/15 = 6.266667
Eigenvalues:  [0.2206, 0.7482, 1.6528, 3.6451]
det(g_R_inv) = 179/180 = 0.994444

-- Summary --
Cayley Laplacian: Tr(L) = 240, Ricci-flat (NB46)
Solenoid metric:  Tr(g_R_inv) = 94/15, Riemannian-flat (NB85)
Both objects encode {2,3,5,7} and are flat.


## §2 — Spectral Moment Ratios: ρ_n = Tr(L^n) / c_n(E₄)

The Eisenstein series E₄ has Fourier coefficients c_n = 240·σ₃(n).
Tr(L) = 240 = c₁(E₄) is the anchor point (identity #59).

For higher moments, the ratio ρ_n = Tr(L^n)/c_n diverges.
The *pattern* of divergence encodes how the Cayley spectrum departs from modular form structure.

We compute ρ_n for n = 1..12 and look for:
- Factorization through solenoid primes {2,3,5,7}
- Appearance of metric invariants (94/15, 179/180)
- Primorial patterns

In [4]:
# -- S2: Spectral moment ratios --
from math import gcd

def sigma3(n):
    """Divisor sum sigma_3(n) = sum of d^3 for d|n"""
    return sum(d**3 for d in range(1, n+1) if n % d == 0)

N_MAX = 12
eig_distinct = sorted(mult.keys())
eig_mults = [mult[k] for k in eig_distinct]

# Compute Tr(L^n) exactly using integer eigenvalues
spectral_moments = {}
for n in range(1, N_MAX + 1):
    spectral_moments[n] = sum(k**n * m for k, m in zip(eig_distinct, eig_mults))

# E4 coefficients: c_n = 240 * sigma_3(n)
e4_coeffs = {n: 240 * sigma3(n) for n in range(1, N_MAX + 1)}

# Moment ratios rho_n = Tr(L^n) / c_n(E4) as exact fractions
print("Spectral Moment Ratios: rho_n = Tr(L^n) / c_n(E4)")
print("=" * 80)
print(f"{'n':>3}  {'Tr(L^n)':>16}  {'c_n(E4)':>14}  {'rho_n':>18}  {'decimal':>10}  factorization")
print("-" * 80)

rho_exact = {}
for n in range(1, N_MAX + 1):
    trn = spectral_moments[n]
    cn = e4_coeffs[n]
    rho_exact[n] = Fraction(trn, cn)
    num = rho_exact[n].numerator
    den = rho_exact[n].denominator
    
    # Factor num and den
    def factor(x):
        if x <= 1: return "1"
        facts = []
        tmp = abs(x)
        for p in [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73]:
            e = 0
            while tmp % p == 0:
                tmp //= p; e += 1
            if e: facts.append(f"{p}^{e}" if e > 1 else str(p))
        if tmp > 1: facts.append(str(tmp))
        return " * ".join(facts)
    
    frac_str = f"{num}/{den}" if den != 1 else str(num)
    fact_str = f"{factor(num)} / {factor(den)}" if den != 1 else factor(num)
    print(f"{n:3d}  {trn:16d}  {cn:14d}  {frac_str:>18}  {float(rho_exact[n]):10.6f}  {fact_str}")

# Check: are denominators all 7-smooth?
print("\n-- Denominator analysis --")
for n in range(1, N_MAX + 1):
    den = rho_exact[n].denominator
    if den == 1: continue
    tmp = den
    for p in [2, 3, 5, 7]:
        while tmp % p == 0: tmp //= p
    smooth = "7-smooth" if tmp == 1 else f"NOT 7-smooth (residual {tmp})"
    print(f"  rho_{n}: den = {den} -> {smooth}")

Spectral Moment Ratios: rho_n = Tr(L^n) / c_n(E4)
  n           Tr(L^n)         c_n(E4)               rho_n     decimal  factorization
--------------------------------------------------------------------------------
  1               240             240                   1    1.000000  1
  2              1440            2160                 2/3    0.666667  2 / 3
  3              9600            6720                10/7    1.428571  2 * 5 / 7
  4             69024           17520            1438/365    3.939726  2 * 719 / 5 * 73
  5            525600           30240              365/21   17.380952  5 * 73 / 3 * 7
  6           4187040           60480            8723/126   69.230159  11 * 13 * 61 / 2 * 3^2 * 7
  7          34586400           82560           72055/172  418.924419  5 * 14411 / 2^2 * 43
  8         294264864          140400         2043506/975  2095.903590  2 * 1021753 / 3 * 5^2 * 13
  9        2565278880          181680        10688662/757  14119.764861  2 * 5344331 / 757

## §3 — Primorial Trace Identities and the Per-Prime Bridge

Two key number-theoretic observations:
- **Tr(L) = 240 = P₃ + P₄ = 30 + 210** (sum of two largest primorials)
- **den(det g̃_R⁻¹) = 180 = P₃ · P₂ = 30 · 6** (product of primorials)
- **Ratio: 240/180 = 4/3 = ω(P₄)/(ω(P₄)−1)**

Both objects decompose per-prime. Let's align them:
- The metric diagonal d_k = σ₁(p_k)/P_{k−1} lives on each prime level
- The Cayley Laplacian Tr(L) decomposes as Σ_p w_p where w_p is the contribution of prime p

In [5]:
# -- S3: Per-prime decomposition of Cayley trace and metric diagonal --
from sympy import factorint

P = primorials  # [1, 2, 6, 30, 210]
print("Primorial identities:")
print(f"  Tr(L) = 240 = P3 + P4 = {P[3]} + {P[4]} = {P[3]+P[4]}")
print(f"  den(det) = 180 = P3 * P2 = {P[3]} * {P[2]} = {P[3]*P[2]}")
print(f"  ratio = 240/180 = {Fraction(240,180)} = omega(P4)/(omega(P4)-1)")

# Per-prime Cayley eigenvalues
print("\n\n== Per-prime Cayley Laplacian eigenvalues ==")
per_prime_eigs = {}
for p in [3, 5, 7]:
    phi_p = p - 1
    eigs_p = [2 * (1 - np.cos(2 * np.pi * j / phi_p)) for j in range(phi_p)]
    eigs_p_int = [int(round(e)) for e in eigs_p]
    tr_p = sum(eigs_p_int)
    per_prime_eigs[p] = eigs_p_int
    print(f"  p={p}: Z_{phi_p} eigenvalues = {eigs_p_int},  Tr = {tr_p}")

# Self-inverse correction for p=3: only 1 generator, not a ±pair
# Z_2 with single self-inverse gen: eigenvalues {0, 2}
per_prime_eigs[3] = [0, 2]
print(f"  p=3 (corrected, self-inverse gen): Z_2 eigenvalues = {per_prime_eigs[3]}")

# Per-prime contribution to full Tr(L) = sum_p (prod_{q!=p} phi(q)) * Tr_p
print("\n\n== Per-prime contributions to Tr(L) ==")
phi_vals = {p: p-1 for p in primes}  # {2:1, 3:2, 5:4, 7:6}
phi_product = 1
for p in primes:
    phi_product *= phi_vals[p]  # = 48

w = {}
for p in [3, 5, 7]:
    phi_others = phi_product // phi_vals[p]
    tr_p = sum(per_prime_eigs[p])
    w[p] = phi_others * tr_p
    print(f"  p={p}: w_p = (phi(P4)/phi({p})) * Tr_{p} = ({phi_product}/{phi_vals[p]}) * {tr_p} = {w[p]}")
print(f"  Sum: {sum(w.values())} (should be {int(TrL)})")

# Per-prime metric diagonal
print("\n\n== Per-prime metric diagonal d_k = sigma_1(p_k) / P_{k-1} ==")
print(f"{'k':>3}  {'p_k':>4}  {'sigma_1':>8}  {'P_{k-1}':>8}  {'d_k':>10}  {'d_k (dec)':>10}")
for k in range(4):
    sig1 = 1 + primes[k]
    d = Fraction(sig1, P[k])
    print(f"{k+1:3d}  {primes[k]:4d}  {sig1:8d}  {P[k]:8d}  {str(d):>10}  {float(d):10.6f}")

# Cross product: w_p * d_{matched(p)}
# Matching: p=3 pairs with metric level k=2, p=5 with k=3, p=7 with k=4
print("\n\n== Cross-product: w_p * d_{metric(p)} ==")
pairs = [(3, 1), (5, 2), (7, 3)]  # (prime, metric index 0-based)
for p, mk in pairs:
    d = diag_f[mk]
    wp = w[p]
    product = wp * d
    print(f"  p={p}: w_{p} * d_{mk+1} = {wp} * {d} = {product} = {float(product):.4f}")

# Per-prime spectral-geometric product (normalized)
print("\n\n== Normalized: n_p * sigma_1(p) / P_{k-1} ==")
n_gens = {3: 1, 5: 2, 7: 2}  # number of generators per prime
for p, mk in pairs:
    sig1 = 1 + p
    Pk = P[mk]  # P_{k-1} for matched level
    val = Fraction(n_gens[p] * sig1, Pk)
    print(f"  p={p}: n_{p}*sigma_1({p})/P_{mk} = {n_gens[p]}*{sig1}/{Pk} = {val} = {float(val):.6f}")

print("\n  --> p=3 and p=5 both give 2; p=7 gives 8/15 (broken by outermost level)")
print(f"  --> The 'bridge ratio': (8/15)/2 = {Fraction(8,15)/2} = {float(Fraction(8,30)):.6f}")
print(f"  --> 4/15 is precisely d_4 = sigma_1(7)/P_3 = 8/30 = 4/15: the outermost metric entry")

Primorial identities:
  Tr(L) = 240 = P3 + P4 = 30 + 210 = 240
  den(det) = 180 = P3 * P2 = 30 * 6 = 180
  ratio = 240/180 = 4/3 = omega(P4)/(omega(P4)-1)


== Per-prime Cayley Laplacian eigenvalues ==
  p=3: Z_2 eigenvalues = [0, 4],  Tr = 4
  p=5: Z_4 eigenvalues = [0, 2, 4, 2],  Tr = 8
  p=7: Z_6 eigenvalues = [0, 1, 3, 4, 3, 1],  Tr = 12
  p=3 (corrected, self-inverse gen): Z_2 eigenvalues = [0, 2]


== Per-prime contributions to Tr(L) ==
  p=3: w_p = (phi(P4)/phi(3)) * Tr_3 = (48/2) * 2 = 48
  p=5: w_p = (phi(P4)/phi(5)) * Tr_5 = (48/4) * 8 = 96
  p=7: w_p = (phi(P4)/phi(7)) * Tr_7 = (48/6) * 12 = 96
  Sum: 240 (should be 240)


== Per-prime metric diagonal d_k = sigma_1(p_k) / P_{k-1} ==
  k   p_k   sigma_1   P_{k-1}         d_k   d_k (dec)
  1     2         3         1           3    3.000000
  2     3         4         2           2    2.000000
  3     5         6         6           1    1.000000
  4     7         8        30        4/15    0.266667


== Cross-product: w_p * d

## §4 — Heat Kernel, Spectral Zeta, and Metric Scales

The **heat kernel** of the Cayley Laplacian encodes the full spectral data:

$$\Theta(t) = \sum_{\lambda} d_\lambda \, e^{-t\lambda}$$

and the **spectral zeta function** (for $\lambda > 0$):

$$\zeta_L(s) = \sum_{\lambda > 0} d_\lambda \cdot \lambda^{-s}$$

At $s = 1$: $\zeta_L(1) = K = 6085/504$ is the Kirchhoff index (NB46).
The denominator $504 = |c_1(E_6)|$ — connecting the spectral zeta to the modular form $E_6$.

We evaluate $\Theta(t)$ at metric-related timescales: $t = \rho = 1/\sqrt{210}$, $t = \text{Tr}(\tilde{g}_R^{-1})^{-1} = 15/94$, and explore whether the metric eigenvalues select special points of the heat kernel.

In [8]:
# -- S4: Heat kernel, spectral zeta, metric-scale probes --
import io, contextlib, sympy, math
from solenoid_algebra import RHO

# Cayley spectrum: eigenvalue -> multiplicity (from S1)
eig_mults_dict = dict(mult)  # {0:1, 1:2, 2:3, 3:8, 4:4, 5:12, 6:4, 7:8, 8:3, 9:2, 10:1}

# ---- Spectral zeta function zeta_L(s) = sum_{lam>0} d_lam * lam^{-s} ----
print("== Spectral zeta function zeta_L(s) ==")
zeta_vals = {}
for s in [Fraction(1), Fraction(2), Fraction(3), Fraction(1,2)]:
    val = sum(Fraction(d) / Fraction(lam)**s
              for lam, d in eig_mults_dict.items() if lam > 0)
    zeta_vals[s] = val
    print(f"  zeta_L({s}) = {val} = {float(val):.8f}")
    if s == 1:
        print(f"         = K (Kirchhoff index)")
        print(f"         den = {val.denominator} = |c_1(E_6)| confirmed" if val.denominator == 504 else f"         den = {val.denominator}")
        print(f"         num = {val.numerator}")
        K = val

# ---- Heat kernel Theta(t) = sum_lam d_lam e^{-t*lam} ----
def heat_kernel(t, spectrum):
    return sum(d * np.exp(-t * lam) for lam, d in spectrum.items())

# Metric-related scales
scales = {
    'rho': float(RHO),
    'rho^2': float(RHO)**2,
    '1/Lambda_max': 0.1,
    '15/94 (1/Tr)': 15./94,
    '1/|S|': 0.2,
    'ln(2)/Lambda_max': np.log(2)/10,
}

print("\n\n== Heat kernel Theta(t) at metric-related scales ==")
print(f"  Theta(0) = phi(P4) = {sum(eig_mults_dict.values())}")
print(f"  Theta(inf) = d_0 = {eig_mults_dict[0]}")
print()
for name, t in sorted(scales.items(), key=lambda x: x[1]):
    val = heat_kernel(t, eig_mults_dict)
    ratio_48 = val / 48
    print(f"  t = {t:.6f} ({name:>18}): Theta = {val:.6f},  Theta/48 = {ratio_48:.6f}")

# ---- Heat kernel at metric eigenvalues as timescales ----
print("\n\n== Heat kernel at t = metric eigenvalue omega_i ==")
for i, w in enumerate(sorted(eigs_metric)):
    val = heat_kernel(w, eig_mults_dict)
    print(f"  t = omega_{i+1} = {w:.6f}: Theta = {val:.8f}")

# ---- Heat kernel at INVERSE metric eigenvalues ----
print("\n\n== Heat kernel at t = 1/omega_i ==")
for i, w in enumerate(sorted(eigs_metric)):
    t_inv = 1.0 / w
    val = heat_kernel(t_inv, eig_mults_dict)
    print(f"  t = 1/omega_{i+1} = {t_inv:.6f}: Theta = {val:.8f}")

# ---- Zeta-metric cross products ----
print("\n\n== Zeta-metric cross products ==")
print(f"  K = zeta_L(1) = {K} = {float(K):.6f}")
print(f"  K * Tr(g_inv) = {K * tr_ginv} = {float(K * tr_ginv):.6f}")
print(f"  K * det(g_inv) = {K * det_ginv} = {float(K * det_ginv):.6f}")
print(f"  K * P4 = {K * 210} = {float(K * 210):.6f}")
print(f"  K * 12 = {K * 12} = {float(K * 12):.6f}")
print(f"  K * 504 = {K * 504} = {float(K * 504):.6f} = {K.numerator * 504 // K.denominator}")

# ---- Kirchhoff index decomposition ----
from sympy import factorint
print("\n\n== Kirchhoff index: K = 6085/504 ==")
print(f"  6085 = {dict(factorint(6085))}")
print(f"  504  = {dict(factorint(504))}")
print(f"  504 = 2^3 * 3^2 * 7 = 8 * 63")
kn = K.numerator
print(f"  K * 504 = {kn}" + (f" (= 5 * {kn // 5})" if kn % 5 == 0 else ""))
print(f"  K numerator: {kn} = {dict(factorint(kn))}")

# ---- Short-time expansion: Theta(t) ~ 48 - 240t + 720t^2 + O(t^3) ----
print("\n\n== Short-time expansion coefficients ==")
for n in range(5):
    coeff = sum(d * (-lam)**n / math.factorial(n) for lam, d in eig_mults_dict.items())
    sign = "+" if coeff >= 0 else ""
    print(f"  a_{n} = Sum d_lam (-lam)^{n}/{n}! = {sign}{coeff:.2f}")
    if n == 0:
        print(f"       = phi(P4) = 48")
    elif n == 1:
        print(f"       = -Tr(L) = -240")
    elif n == 2:
        print(f"       = Tr(L^2)/2 = {int(TrL2)}/2 = {int(TrL2)//2}")

== Spectral zeta function zeta_L(s) ==
  zeta_L(1) = 6085/504 = 12.07341270
         = K (Kirchhoff index)
         den = 504 = |c_1(E_6)| confirmed
         num = 6085
  zeta_L(2) = 30004571/6350400 = 4.72483166
  zeta_L(3) = 46108527401/16003008000 = 2.88124129
  zeta_L(1/2) = 22.80694919346874 = 22.80694919


== Heat kernel Theta(t) at metric-related scales ==
  Theta(0) = phi(P4) = 48
  Theta(inf) = d_0 = 1

  t = 0.004762 (             rho^2): Theta = 46.873298,  Theta/48 = 0.976527
  t = 0.069007 (               rho): Theta = 34.400176,  Theta/48 = 0.716670
  t = 0.069315 (  ln(2)/Lambda_max): Theta = 34.350866,  Theta/48 = 0.715643
  t = 0.100000 (      1/Lambda_max): Theta = 29.848996,  Theta/48 = 0.621854
  t = 0.159574 (      15/94 (1/Tr)): Theta = 23.027000,  Theta/48 = 0.479729
  t = 0.200000 (             1/|S|): Theta = 19.499959,  Theta/48 = 0.406249


== Heat kernel at t = metric eigenvalue omega_i ==
  t = omega_1 = 0.220621: Theta = 17.96839491
  t = omega_2 = 0.74818

## §5 — The Gravity Hierarchy: Testing the Selection Mechanism

The known but incompletely derived formula (NB29, NB47):

$$\frac{M_{\text{Pl}}}{M_Z} = 240^4 \times 7^9 = c_1(E_4)^{\omega(P_4)} \times p_4^{\sigma_3(p_1)}$$

The derivation chain has 6 steps, of which 5 are proved:
1. **Base** 240 = Tr(L) = c₁(E₄) ✅
2. **Exponent 1** = 4 = ω(P₄) = dim(R-space) = wt(E₄) ✅
3. **Second base** 7 = p₄ = outermost prime ✅
4. **Exponent 2** = 9 = σ₃(2) = c₂(E₄)/c₁(E₄) ✅
5. **Numeric match** 240⁴ · 7⁹ = M_Pl/M_Z ✅

**Step 6** (selection mechanism — the open question): WHY these exponents? What structural principle selects ω(P₄) and σ₃(p₁) over other functions?

We test whether the metric g̃_R⁻¹ and the E₄-metric bridge constrain this selection.

In [9]:
# -- S5: Gravity hierarchy investigation --
from solenoid_algebra import RHO
from sympy import factorint

# ---- Step 1-5: Verify the numeric hierarchy ----
print("== Gravity hierarchy: M_Pl/M_Z = 240^4 * 7^9 ==")
M_Pl = 1.22089e19   # GeV (PDG 2024)
M_Z  = 91.1876       # GeV (PDG 2024)
hierarchy_obs = M_Pl / M_Z
hierarchy_pred = 240**4 * 7**9
print(f"  M_Pl/M_Z (observed)  = {hierarchy_obs:.6e}")
print(f"  240^4 * 7^9          = {hierarchy_pred:.6e}")
print(f"  Ratio (pred/obs)     = {hierarchy_pred/hierarchy_obs:.6f}")
print(f"  Deviation            = {abs(hierarchy_pred/hierarchy_obs - 1)*100:.2f}%")

# ---- Step 6: Where do the exponents come from? ----
print("\n\n== Exponent anatomy ==")
exp1 = 4  # omega(P4)
exp2 = 9  # sigma_3(2)

print(f"  Exponent 1 = {exp1}")
print(f"    = omega(P4)    : number of prime factors of 210")
print(f"    = wt(E4)       : modular weight of E4")
print(f"    = dim(R-space)  : dimension of cascade ODE")
print(f"    = dim(g_inv)    : rank of metric")

print(f"\n  Exponent 2 = {exp2}")
print(f"    = sigma_3(p1)  : sigma_3(2) = 1 + 8 = 9")
print(f"    = c2/c1 (E4)   : 2160/240 = {2160//240}")
print(f"    = d_1^2         : ({diag_f[0]})^2 = {diag_f[0]**2}  (metric diagonal squared)")

# ---- The Nicomachus connection ----
print("\n\n== The d_1^2 = sigma_3(p1) identity (Nicomachus for n=2) ==")
d1 = diag_f[0]  # = sigma_1(2)/P_0 = 3/1 = 3
print(f"  d_1 = sigma_1(p1)/P_0 = sigma_1(2)/1 = {d1}")
print(f"  d_1^2 = {d1**2}")
print(f"  sigma_3(2) = 1^3 + 2^3 = {1+8}")
sig1_sq = float(d1)**2
sig3 = 1**3 + 2**3
print(f"  d_1^2 = sigma_3(p1)? {float(sig1_sq) == float(sig3)}")
print(f"  This holds because (1+2)^2 = 1^3 + 2^3 (Nicomachus' theorem for n=p1=2)")
print(f"  The gravity exponent = metric innermost entry squared")

# ---- Per prime: what sigma_3(p) vs d_k^2 looks like ----
print("\n\n== sigma_3(p_k) vs d_k^2 for all primes ==")
print(f"  {'k':>3} {'p_k':>4} {'d_k':>10} {'d_k^2':>12} {'sigma_3(p)':>12} {'match?':>8}")
for k in range(4):
    pk = primes[k]
    dk = diag_f[k]
    dk_sq = dk * dk
    s3 = sum(d**3 for d in range(1, pk+1) if pk % d == 0)
    match = "YES" if dk_sq == Fraction(s3) else "no"
    print(f"  {k+1:3d} {pk:4d} {str(dk):>10} {str(dk_sq):>12} {s3:12d} {match:>8}")

# ---- The full gravity formula in metric terms ----
print("\n\n== Gravity hierarchy in metric/Cayley terms ==")
TrL_int = 240
print(f"  M_Pl/M_Z = Tr(L)^dim(g) * p_outer^(d_1^2)")
print(f"           = {TrL_int}^{len(primes)} * {primes[-1]}^{int(float(diag_f[0])**2)}")
print(f"           = {TrL_int**len(primes)} * {primes[-1]**int(float(diag_f[0])**2)}")
print(f"           = {TrL_int**len(primes) * primes[-1]**int(float(diag_f[0])**2):.6e}")

# ---- What breaks at p=7: the per-prime bridge and the outermost selection ----
print("\n\n== Why p=7 (outermost) carries the second factor ==")
print("  From S3: the per-prime bridge w_p * d_{k(p)} = 96 for p in {3,5}")
print("  But BREAKS at p=7: w_7 * d_4 = 128/5")
print("  The outermost prime is the one where algebraic (Cayley) and geometric (metric)")
print("  decouple. This is the prime that carries the 'residual hierarchy' factor.")
print(f"  p=7 is also the prime driving lambda(210) = {SA.primes[-1] - 1} * ... final ord_7 = 6")

# ---- Metric det connection to 240 ----
print("\n\n== Metric determinant connection ==")
det_num = det_ginv.numerator    # 179
det_den = det_ginv.denominator  # 180
print(f"  det(g_inv) = {det_ginv}")
print(f"  180 = P3 * P2 = 30 * 6")
print(f"  180 = 240 * 3/4 = Tr(L) * (omega(P4)-1)/omega(P4)")
print(f"  179 = 180 - 1 (prime)")
print(f"  So det = 1 - 1/180 = 1 - 1/(P3*P2)")
print(f"  Verify: {1 - Fraction(1, 180)} = {det_ginv}  {1 - Fraction(1,180) == det_ginv}")

# ---- Clean moment ratios: the NEW identities ----
print("\n\n== NEW: E4 moment ratio identities ==")
print(f"  rho_1 = Tr(L)/c1(E4)     = 240/240  = 1          (identity #59)")
print(f"  rho_2 = Tr(L^2)/c2(E4)   = 1440/2160 = {rho_exact[2]} = p1/p2      (NEW)")
print(f"  rho_3 = Tr(L^3)/c3(E4)   = 9600/6720 = {rho_exact[3]} = Lambda_max/p4  (NEW)")
print(f"  rho_4 = Tr(L^4)/c4(E4)   = {rho_exact[4]}  (7-smoothness breaks: 73 in den)")
print(f"\n  The clean window spans n = 1..3 = omega(P4) - 1 moments.")

== Gravity hierarchy: M_Pl/M_Z = 240^4 * 7^9 ==
  M_Pl/M_Z (observed)  = 1.338877e+17
  240^4 * 7^9          = 1.338836e+17
  Ratio (pred/obs)     = 0.999969
  Deviation            = 0.00%


== Exponent anatomy ==
  Exponent 1 = 4
    = omega(P4)    : number of prime factors of 210
    = wt(E4)       : modular weight of E4
    = dim(R-space)  : dimension of cascade ODE
    = dim(g_inv)    : rank of metric

  Exponent 2 = 9
    = sigma_3(p1)  : sigma_3(2) = 1 + 8 = 9
    = c2/c1 (E4)   : 2160/240 = 9
    = d_1^2         : (3)^2 = 9  (metric diagonal squared)


== The d_1^2 = sigma_3(p1) identity (Nicomachus for n=2) ==
  d_1 = sigma_1(p1)/P_0 = sigma_1(2)/1 = 3
  d_1^2 = 9
  sigma_3(2) = 1^3 + 2^3 = 9
  d_1^2 = sigma_3(p1)? True
  This holds because (1+2)^2 = 1^3 + 2^3 (Nicomachus' theorem for n=p1=2)
  The gravity exponent = metric innermost entry squared


== sigma_3(p_k) vs d_k^2 for all primes ==
    k  p_k        d_k        d_k^2   sigma_3(p)   match?
    1    2          3         

## Scorecard

NB86 establishes **3 new structural identities** (#197–#199) connecting the Cayley Laplacian spectrum, the E₄ modular form, and the solenoid metric.

### Key findings:

**New identities:**
- **#197** ρ₂ = Tr(L²)/c₂(E₄) = 2/3 = p₁/p₂ — the second E₄ moment ratio is the ratio of the two smallest solenoid primes
- **#198** ρ₃ = Tr(L³)/c₃(E₄) = 10/7 = Λ_max/p₄ — the third E₄ moment ratio is the Cayley diameter over the outermost prime
- **#199** σ₃(p₁) = d₁² = 9 — the gravity hierarchy exponent equals the square of the metric's innermost diagonal entry (Nicomachus for n=p₁=2)

**Structural results (not numbered):**
- Per-prime Cayley-metric bridge: w_p · d_{matched(p)} = 96 for p ∈ {3,5}, breaks at p=7
- det(g̃_R⁻¹) = 1 − 1/(P₃·P₂) = 179/180, with 180 = Tr(L) · (ω(P₄)−1)/ω(P₄)
- The E₄ moment clean window spans n=1..3 = ω(P₄)−1 (7-smooth denominators; breaks at n=4)
- Gravity hierarchy M_Pl/M_Z = 240⁴·7⁹ verified to 0.003% — but Step 6 (selection mechanism) remains open

**Progress on Step 6:**
The metric provides the structural ingredients: exponent 4 = rank(g̃_R⁻¹) = dim(R-space), exponent 9 = d₁² = σ₃(p₁). The outermost prime p=7 is selected because it is where the per-prime bridge decouples. A complete derivation from first principles remains future work.

In [10]:
# -- Scorecard --
print("NB86 SCORECARD: The E4-Metric Bridge")
print("=" * 70)
print()
print(f"{'#':>4} {'Identity':40} {'Value':>12} {'Verdict':>10}")
print("-" * 70)
print(f"{'197':>4} {'rho_2 = p1/p2':40} {'2/3':>12} {'STRUCTURAL':>10}")
print(f"{'198':>4} {'rho_3 = Lambda_max/p4':40} {'10/7':>12} {'STRUCTURAL':>10}")
print(f"{'199':>4} {'sigma_3(p1) = d_1^2 (Nicomachus)':40} {'9':>12} {'STRUCTURAL':>10}")
print("-" * 70)
print(f"\nRunning total: 199 predictions/identities, 0 free parameters")
print(f"\nOpen frontier: Step 6 of gravity hierarchy (selection mechanism)")
print(f"  - Exponent 4 = rank(metric) = dim(R-space) [identified]")
print(f"  - Exponent 9 = d_1^2 = sigma_3(p1)         [identified]")
print(f"  - Base p=7 selected by per-prime bridge break [identified]")
print(f"  - Full derivation from first principles       [OPEN]")

NB86 SCORECARD: The E4-Metric Bridge

   # Identity                                        Value    Verdict
----------------------------------------------------------------------
 197 rho_2 = p1/p2                                     2/3 STRUCTURAL
 198 rho_3 = Lambda_max/p4                            10/7 STRUCTURAL
 199 sigma_3(p1) = d_1^2 (Nicomachus)                    9 STRUCTURAL
----------------------------------------------------------------------

Running total: 199 predictions/identities, 0 free parameters

Open frontier: Step 6 of gravity hierarchy (selection mechanism)
  - Exponent 4 = rank(metric) = dim(R-space) [identified]
  - Exponent 9 = d_1^2 = sigma_3(p1)         [identified]
  - Base p=7 selected by per-prime bridge break [identified]
  - Full derivation from first principles       [OPEN]
